# E0: Linear Probing — CheXpert Baseline

Train only the classification head with frozen DINOv2 backbone. This is Experiment E0 — the linear probing baseline.

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, mount Google Drive.
"""
import os
import subprocess

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(["pip", "install", "-e", ".", "-r", "requirements.txt", "-q"], check=True)

from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Configuration

Load CheXpert benchmark dataset and PEFT experiment configuration profiles.

In [ ]:
"""
Load configuration files for CheXpert dataset and PEFT experiments.
"""
from typing import Any
from src.utils.config import load_config

cfg: dict[str, Any] = load_config("configs/chexpert.yaml")
peft_cfg: dict[str, Any] = load_config("configs/peft/lora.yaml")
print("CheXpert Configuration:", cfg)
print("PEFT LoRA Reference Configuration:", peft_cfg)

## 2. Build Datasets

Instantiate data augmentation and normalization transforms, and construct training and validation dataloaders.

In [ ]:
"""
Construct PyTorch data transforms and DataLoaders for CheXpert training and validation.
"""
from src.data import build_chexpert_dataloaders, build_transforms

train_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=True)
val_tf = build_transforms(224, cfg["preprocessing"]["normalize_mean"], cfg["preprocessing"]["normalize_std"], augment=False)
train_loader, val_loader = build_chexpert_dataloaders(cfg, train_tf, val_tf)
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

## 3. Build Model

Load DINOv2 ViT-B/14, completely freeze its backbone parameters, and append a trainable multi-label classification head.

In [ ]:
"""
Instantiate frozen DINOv2 ViT-B/14 backbone and trainable multi-label classification head.
"""
import torch
import torch.nn as nn
from src.models import count_trainable_parameters, freeze_backbone, load_backbone, MultiLabelHead

backbone, processor = load_backbone()
freeze_backbone(backbone)
head = MultiLabelHead(in_features=768, num_labels=14)

class LinearProbeModel(nn.Module):
    """
    End-to-end model combining frozen DINOv2 backbone and trainable linear classification head.
    Forward pass extracts CLS token representation and passes it through the head.
    """
    def __init__(self, backbone: nn.Module, head: nn.Module) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Extracts CLS token from backbone and returns classification logits.
        """
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.head(cls_token)

model = LinearProbeModel(backbone, head)
trainable, total = count_trainable_parameters(model)
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

## 4. Training

Train the linear probe classification head using `MultiLabelBCELoss` and the AdamW optimizer for 10 epochs.

In [ ]:
"""
Execute training loop for linear probe using MultiLabelBCELoss, AdamW optimizer, and checkpoint persistence.
"""
import torch
from torch.optim import AdamW
from src.training.losses import MultiLabelBCELoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = MultiLabelBCELoss()
optimizer = AdamW(head.parameters(), lr=5e-3, weight_decay=1e-4)

epochs: int = 10
print(f"Training linear probe for {epochs} epochs on device: {device}")

for epoch in range(epochs):
    model.train()
    running_loss: float = 0.0
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        logits = model(pixel_values)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss: float = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1:02d}/{epochs:02d}] - Loss: {avg_loss:.4f}")

checkpoint_save_path: str = f"{CHECKPOINT_DIR}/linear_probe_chexpert.pt"
torch.save(head.state_dict(), checkpoint_save_path)
print(f"Trained classification head checkpoint saved to: {checkpoint_save_path}")

## 5. Evaluation

Evaluate the trained linear probe on the validation split and calculate mean per-label AUROC.

In [ ]:
"""
Evaluate linear probe model on validation split and report mean AUROC.
"""
from typing import Any
from src.evaluation import Evaluator

evaluator = Evaluator(task="multilabel", label_names=cfg["dataset"]["label_names"])
results: dict[str, Any] = evaluator.evaluate(model, val_loader, device="cuda" if torch.cuda.is_available() else "cpu")
print(f"Mean AUROC: {results['mean_auroc']:.4f}")
for label, score in results.get("per_label_auroc", {}).items():
    print(f"  {label:<28}: {score:.4f}")

## 6. Log Experiment

Persist experiment metrics and run metadata to the centralized experiment tracking log.

In [ ]:
"""
Log experiment run metadata and validation results into unified log.
"""
from src.utils.logging_utils import ExperimentLogger

logger = ExperimentLogger()
logger.log(
    run_id="lp_chexpert",
    dataset="CheXpert",
    backbone="DINOv2 ViT-B/14",
    method="LinearProbing",
    val_auroc=results["mean_auroc"],
)
print("Experiment log record successfully appended.")

## 7. Visualize Results

Plot a bar chart of per-label AUROC scores alongside the dataset mean benchmark line.

In [ ]:
"""
Generate and display per-label AUROC bar chart for CheXpert linear probing.
"""
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

per_label_dict = results.get("per_label_auroc", {})
plot_df = pd.DataFrame({"label": list(per_label_dict.keys()), "auroc": list(per_label_dict.values())})
plot_df = plot_df.sort_values(by="auroc", ascending=False)

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_df, x="auroc", y="label", color="steelblue", ax=ax)
ax.axvline(x=results["mean_auroc"], color="crimson", linestyle="--", linewidth=2, label=f"Mean AUROC: {results['mean_auroc']:.4f}")
ax.set_title("E0: Linear Probing Baseline — Per-Label AUROC (CheXpert)", fontsize=14, fontweight="bold")
ax.set_xlabel("Validation AUROC", fontsize=12)
ax.set_ylabel("Pathology Condition", fontsize=12)
ax.set_xlim(0.5, 1.0)
ax.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.show()